In [5]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/Depi_Graduation_Project/step_1 _audio_processing&ASR')

Mounted at /content/drive


In [10]:
!pip install -q faster-whisper pydub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 78.6 MB/s eta 0:00:00


In [6]:
!apt-get install -q -y ffmpeg

Reading package lists...^C


In [7]:
video_path='/content/drive/MyDrive/Depi_Graduation_Project/step_1 _audio_processing&ASR/input_video2.mp4'

In [8]:
import os
def get_audio_from_video(video_path):
  audio_path="raw_audio.wav"
  !ffmpeg -i "{video_path}" -ar 16000 -ac 1 -c:a pcm_s16le "{audio_path}" -y -q:a 0
  return audio_path
audio_result = get_audio_from_video("input_video2.mp4")
print(f"Audio is ready here: {audio_result}")



ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [11]:
from faster_whisper import WhisperModel
model = WhisperModel("large-v3", device="cuda", compute_type="float16")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [12]:

def speech_to_text(audio_file):

    print("Extracting text plz wait... ")
    segments, info = model.transcribe(audio_file, beam_size=5, word_timestamps=True)
    results = []
    for segment in segments:
        results.append({
            "start": segment.start,
            "end": segment.end,
            "text": segment.text.strip()
        })
        print(f"[{segment.start:.2f}s -> {segment.end:.2f}s] {segment.text}")

    return results, info.language
final_text = speech_to_text("raw_audio.wav")

Extracting text plz wait... 
[0.00s -> 3.26s]  Today's networks face thousands of threats every single moment.
[4.08s -> 6.92s]  Now imagine being able to see it all from inside the system itself,
[7.90s -> 13.18s]  watching every bite move, every unusual activity, before it becomes an attack.
[13.74s -> 17.26s]  In the real world, human response time isn't always enough.
[17.80s -> 20.00s]  Attacks slip through in fractions of a second.
[20.14s -> 25.00s]  And just one breach can shut down an entire system and cost companies millions.
[25.48s -> 28.50s]  That's why we introduced the AI-powered DevOps system,
[28.50s -> 32.40s]  an intelligent platform that merges the power of artificial intelligence
[32.40s -> 38.44s]  with fully automated DevOps workflows to monitor, analyze, and protect networks in real time.
[38.58s -> 44.36s]  The system begins by collecting live network data using tools like Prometheus and Grafana,
[44.54s -> 49.46s]  then applies machine learning models built wi

In [13]:
segments, info = model.transcribe("raw_audio.wav")
detected_lang = info.language
print(f"language is: {detected_lang}")

language is: en


In [14]:
import json
def save_results(results, language):
    output_data = {
        "detected_language": language,
        "segments": results
    }

    with open("text_data.json", "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=4, ensure_ascii=False)

    print("Done...")

save_results(final_text, detected_lang)

Done...
